<hr>
<center> Coding Exercise for Control Systems<br>
<b> Exercise 1: Representations </b> <br>
Prof. Dr. Florian Dörfler <br>
Automatic Control Laboratory, ETH Zurich </center>
<hr>

In this notebook, we learn about the basic Python commands for system representations and experiement with them.

<blockquote>
<b>Activity:</b> Execute the code cells below so that the necessary libraries get imported.
</blockquote>

In [ ]:
!pip install control

In [2]:
import numpy as np # the standard library for numerics, vectors, matrices
import control as ct # the standard library for basic operations for analysis and design of feedback control systems
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations

<b> C 1.1: State-Space Representations of Linear Systems </b>

Consider the continuous-time state-space model of a system 1:
\begin{equation*}
\dot x = \begin{bmatrix} \alpha & -\beta \\ \beta & \alpha \end{bmatrix} x \tag{1}
\end{equation*}

<blockquote>
<b>Activity:</b> Complete the code fragment below and create the state-space object.
</blockquote>

In [ ]:
# define parameters
a = -1
b = 2

# define the state-space matrices
A1 = np.array([[a, -b],[b, a]])
B1 = np.zeros((2,1))
C1 = np.eye(2)
D1 = np.zeros((2,1))

# create state-space object
sys1_ss = ct.ss(A1, B1, C1, D1)

<blockquote>
<b>Activity:</b> Plot the initial response of the system for x=(1,1).
</blockquote>

In [ ]:
# initial condition
x0 = [1,1]

# compute the initial response (outputs are the states because C=I)
t_init_1 = np.linspace(0,10,201)
t_init_1, x_init_1 = ct.initial_response(sys1_ss, T=t_init_1, X0=x0)
x1_init_1 = x_init_1[0]
x2_init_1 = x_init_1[1]

# plots
plt.plot(t_init_1, x1_init_1, label='$x_1$')
plt.plot(t_init_1, x2_init_1, label='$x_2$')
plt.xlabel("seconds")
plt.title("initial response")
plt.legend()
plt.grid()
plt.show()

<blockquote>
<b>Activity:</b> Simulate the system different a and b (positive, negative, zero). What do you notice about the system behavior?
</blockquote>

<i> Answer on system comparison: . . . </i>

Consider the following system 2 with an input:
\begin{equation*}
\dot x = \begin{bmatrix} 0 & 1 \\ -1 & -1 \end{bmatrix} x + \begin{bmatrix} 0 \\ 4 \end{bmatrix} u \tag{2}
\end{equation*}

\begin{equation*}
y = \begin{bmatrix} 1 & 0 \end{bmatrix} x 
\end{equation*}

In [ ]:
# system matrices
A2 = np.array([[0,1],[-1,-1]])
B2 = np.array([[0],[4]])
C2 = np.array([[1,0]])
D2 = np.array([[0]])

# create state-space object
sys2_ss = ct.ss(A2, B2, C2, D2)

Ellipsis

<blockquote>
<b>Activity:</b> Plot the step response of the system.
</blockquote>

In [ ]:
# compute the step response
t_step_2 = np.linspace(0,10,201)
t_step_2, y_step_2 = ct.step_response(sys2_ss, T=t_step_2)

# plots
plt.plot(t_step_2, y_step_2, label='$y_2$')
plt.xlabel("seconds")
plt.title("step response")
plt.legend()
plt.grid()
plt.show()

Consider the discrete representation of system 2d with a sample time of $\Delta t$:
\begin{equation*}
x[k+1] = \begin{bmatrix} 1 & \Delta t \\ -\Delta t & 1-\Delta t \end{bmatrix} x[k] + \begin{bmatrix} 0 \\ 4\Delta t \end{bmatrix} u[k] \tag{2d}
\end{equation*}

\begin{equation*}
y[k] = \begin{bmatrix} 1 & 0 \end{bmatrix} x[k] 
\end{equation*}

<blockquote>
<b>Activity:</b> Define the state-space object and plot the step response of the system over that of the continuous system.
</blockquote>

In [ ]:
# sample time
dt = 0.5

# system matrices
A2d = np.array([[1, dt],[-dt, 1-dt]])
B2d = np.array([[0],[4*dt]])
C2d = np.array([[1,0]])
D2d = np.array([[0]])

# create discrete-time state-space object (specify dt)
sys2d_ss = ct.ss(A2d, B2d, C2d, D2d, dt=dt)

# compute the step response for discrete system on matching time grid
t_step_2d = np.arange(0, 21) * dt
t_step_2d, y_step_2d = ct.step_response(sys2d_ss, T=t_step_2d)

# plots
plt.plot(t_step_2, y_step_2, label='$y_2$')
plt.plot(t_step_2d, y_step_2d, 'o-', label='$y_2d$')
plt.xlabel("seconds")
plt.title("step response")
plt.legend()
plt.grid()
plt.show()

<blockquote>
<b>Activity:</b>
What differences do you notice between the continuous and discrete models? What happens if you change the sample time?
</blockquote>

<i> Sample time impact on continuous and discrete models: </i> ...

<b> C 1.2: Linearization impacts on nonlinear systems </b>

Consider a nonlinear continuous-time system 3 and its linearized counterpart 4 centered at stable equilibrium $x^\ast = (0,0)$
\begin{equation}
\dot x = \begin{bmatrix} \dot x_1 \\ \dot x_2 \end{bmatrix} = \begin{bmatrix} x_2 \\ -x_1 - x_2^3 \end{bmatrix} \tag{3}
\end{equation}

\begin{equation}
\Delta \dot x_{eq} = \begin{bmatrix} 0 & 1 \\ -1 & 0 \end{bmatrix} \Delta x_{eq} \tag{4}
\end{equation}

where $\Delta x_{eq} = x - x_{eq}$.

<blockquote>
<b>Activity:</b>
Complete the functions and create the nonlinear system object.
</blockquote>

In [ ]:
# define the nonlinear update function
def update(t, x, u, params):
    x1dot = x[1]
    x2dot = -x[0] - x[1]**3
    return np.array([x1dot, x2dot])

# pass through all states as output
def output(t, x, u, params):
    y = x
    return y

# create a NonlinearIOSystem for simulation
try:
    sys3_nl = ct.NonlinearIOSystem(update, output, name='sys3_nl', states=['x1','x2'], outputs=['y1','y2'])
except Exception:
    # fallback if the helper is named differently in the installed control package
    sys3_nl = ct.nlsys(update, output, name='sys3_nl', states=['x1','x2'], outputs=['y1','y2'])

<blockquote>
<b>Activity:</b>
Create the state-space object for the linearized model.
</blockquote>

In [ ]:
# system matrices (linearization around the origin)
A4 = np.array([[0,1],[-1,0]])
B4 = np.zeros((2,1))
C4 = np.eye(2)
D4 = np.zeros((2,1))

# create state-space object
sys4_ss = ct.ss(A4, B4, C4, D4)

<blockquote>
<b>Activity:</b>
Complete the code block below to generate and plot the initial responses of both the nonlinear system and the linearization for a certain initial condition.
</blockquote>

In [ ]:
# initial condition
x0 = [-0.1, 0.1]

# compute the nonlinear response by integrating the ODE
from scipy.integrate import solve_ivp
t_init_3 = np.linspace(0, 10, 501)
def f(t, x):
    return [x[1], -x[0] - x[1]**3]
sol = solve_ivp(f, [t_init_3[0], t_init_3[-1]], x0, t_eval=t_init_3)
x_init_3 = sol.y

# extract states
x1_init_3 = x_init_3[0]
x2_init_3 = x_init_3[1]

# compute the linearized initial response (outputs are states)
t_init_4, x_init_4 = ct.initial_response(sys4_ss, T=t_init_3, X0=x0)

# extract states
x1_init_4 = x_init_4[0]
x2_init_4 = x_init_4[1]

# plots
fig, ax = plt.subplots(2,1, sharex=True)
ax[0].plot(t_init_3, x1_init_3, label='$x_1^{true}$')
ax[0].plot(t_init_4, x1_init_4, '--', label='$x_1^{lin}$')
ax[0].legend()
ax[0].grid()
ax[1].plot(t_init_3, x2_init_3, label='$x_2^{true}$')
ax[1].plot(t_init_4, x2_init_4, '--', label='$x_2^{lin}$')
ax[1].legend()
ax[1].grid()
plt.xlabel("seconds")
plt.suptitle("initial response")
plt.show()

<blockquote>
<b>Activity:</b>
Run the simulation for different initial conditions. What do you notice as the initial condition values trend away from (0,0)?
</blockquote>

<i> Impact of distance to the equilibrium point: </i> ...